# Tracking Lease Contract IDs and Addresses

Remember to record the linkage between the leasing contract ID and the address for each property. 

In [7]:
import pandas as pd

df = pd.read_csv("../../data/1_derived/1_sf_properties_for_geocoding.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (9352, 11)


,LeaseCompId,PropertyId,PropertyName,full_address,Address.DeliveryAddress,Address.Locality,Address.Subdivision,Address.PostalCode,Address.County,Market,Submarket
0,167100221.0,36156.0,NaN,"445 Bush St, San Francisco, CA 94108, United S...",445 Bush St,San Francisco,CA,94108.0,San Francisco,San Francisco,Union Square
1,162545361.0,36157.0,New Montgomery Tower,"33 New Montgomery St, San Francisco, CA 94105,...",33 New Montgomery St,San Francisco,CA,94105.0,San Francisco,San Francisco,South Financial District
2,70221222.0,36158.0,115 Sansome St,"115 Sansome St, San Francisco, CA 94104, Unite...",115 Sansome St,San Francisco,CA,94104.0,San Francisco,San Francisco,Financial District
3,133318191.0,36159.0,Second Street Square,"501 2nd St, San Francisco, CA 94107, United St...",501 2nd St,San Francisco,CA,94107.0,San Francisco,San Francisco,Rincon/South Beach
4,70087748.0,36160.0,NaN,"110 Sutter St, San Francisco, CA 94104, United...",110 Sutter St,San Francisco,CA,94104.0,San Francisco,San Francisco,Financial District


In [8]:
# Check for leasing contract ID and address columns
cols = df.columns.tolist()

leasing_id_candidates = [c for c in cols if any(kw in c.lower() for kw in ["contract", "lease", "leasing", "id"])]
address_candidates    = [c for c in cols if any(kw in c.lower() for kw in ["address", "addr"])]

print("All columns:", cols)
print()
print("Leasing/contract/ID candidates:", leasing_id_candidates if leasing_id_candidates else "NONE FOUND")
print("Address candidates            :", address_candidates    if address_candidates    else "NONE FOUND")

All columns: ['LeaseCompId', 'PropertyId', 'PropertyName', 'full_address', 'Address.DeliveryAddress', 'Address.Locality', 'Address.Subdivision', 'Address.PostalCode', 'Address.County', 'Market', 'Submarket']

Leasing/contract/ID candidates: ['LeaseCompId', 'PropertyId']
Address candidates            : ['full_address', 'Address.DeliveryAddress', 'Address.Locality', 'Address.Subdivision', 'Address.PostalCode', 'Address.County']


In [9]:
# Standardize full_address for Census Geocoder and show percent changed
import re

def normalize_address_range(addr):
    if not addr:
        return addr
    # Match a leading house-number range like "1234-5678" and keep only the first number
    addr = re.sub(r'^(\d+)-\d+(\s)', r'\1\2', addr)
    return addr

def standardize_address(addr):
    if pd.isnull(addr):
        return ""
    addr = addr.strip().upper()
    addr = re.sub(r'\s+', ' ', addr)  # Remove extra spaces
    addr = re.sub(r'[^\w\s,]', '', addr)  # Remove punctuation except comma
    abbreviations = {
        " STREET ": " ST ",
        " AVENUE ": " AVE ",
        " ROAD ": " RD ",
        " BOULEVARD ": " BLVD ",
        " DRIVE ": " DR ",
        " COURT ": " CT ",
        " PLACE ": " PL ",
        " LANE ": " LN ",
        " TERRACE ": " TER ",
        " PARKWAY ": " PKWY ",
        " SQUARE ": " SQ ",
        " SUITE ": " STE ",
        " APARTMENT ": " APT ",
        " UNIT ": " UNIT ",
    }
    for k, v in abbreviations.items():
        addr = addr.replace(k, v)
    return addr

# Normalize ranges first, then standardize
df['full_address_std'] = df['full_address'].apply(normalize_address_range).apply(standardize_address)
changed_count = (df['full_address'] != df['full_address_std']).sum()
total_count = df['full_address'].notnull().sum()
percent_changed = 100 * changed_count / total_count if total_count > 0 else 0
print(f"Percent of addresses changed: {percent_changed:.2f}%")
print(df[['full_address', 'full_address_std']].head())


Percent of addresses changed: 100.00%
                                        full_address  \
0  445 Bush St, San Francisco, CA 94108, United S...   
1  33 New Montgomery St, San Francisco, CA 94105,...   
2  115 Sansome St, San Francisco, CA 94104, Unite...   
3  501 2nd St, San Francisco, CA 94107, United St...   
4  110 Sutter St, San Francisco, CA 94104, United...   

                                    full_address_std  
0  445 BUSH ST, SAN FRANCISCO, CA 94108, UNITED S...  
1  33 NEW MONTGOMERY ST, SAN FRANCISCO, CA 94105,...  
2  115 SANSOME ST, SAN FRANCISCO, CA 94104, UNITE...  
3  501 2ND ST, SAN FRANCISCO, CA 94107, UNITED ST...  
4  110 SUTTER ST, SAN FRANCISCO, CA 94104, UNITED...  


In [10]:
# Count how many raw addresses had a range (normalization already applied above)
range_changed = df['full_address'].fillna('').apply(
    lambda addr: bool(re.match(r'^\d+-\d+\s', addr))
).sum()
print(f"Addresses with ranges normalized: {range_changed}")
print(df[['full_address', 'full_address_std']].head())


Addresses with ranges normalized: 3097
                                        full_address  \
0  445 Bush St, San Francisco, CA 94108, United S...   
1  33 New Montgomery St, San Francisco, CA 94105,...   
2  115 Sansome St, San Francisco, CA 94104, Unite...   
3  501 2nd St, San Francisco, CA 94107, United St...   
4  110 Sutter St, San Francisco, CA 94104, United...   

                                    full_address_std  
0  445 BUSH ST, SAN FRANCISCO, CA 94108, UNITED S...  
1  33 NEW MONTGOMERY ST, SAN FRANCISCO, CA 94105,...  
2  115 SANSOME ST, SAN FRANCISCO, CA 94104, UNITE...  
3  501 2ND ST, SAN FRANCISCO, CA 94107, UNITED ST...  
4  110 SUTTER ST, SAN FRANCISCO, CA 94104, UNITED...  


In [11]:
# Drop original full_address column after standardization
if 'full_address' in df.columns:
    df.drop(columns=['full_address'], inplace=True)
print("Dropped original full_address column.")

Dropped original full_address column.


In [12]:
# Save standardized DataFrame
output_path = '../../data/1_derived/2_sf_properties_for_geocoding_standardized.csv'
df.to_csv(output_path, index=False)
print(f"Saved standardized DataFrame to {output_path}")

Saved standardized DataFrame to ../../data/1_derived/2_sf_properties_for_geocoding_standardized.csv
